In [ ]:
import cv2
import os
import pandas as pd
from tqdm import tqdm
os.environ['QT_QPA_PLATFORM'] = 'xcb'

# Parameters
NUM_FOLDERS = 30
NUM_FRAMES = 30
FRAME_WIDTH = 640
FRAME_HEIGHT = 480

# Create main dataset directory
dataset_dir = './Gesture Dataset/Pinch'
if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir)

# Initialize CSV with simplified structure
csv_file = 'dataset_metadata.csv'
if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    last_folder = df['folder_name'].iloc[-1]
    start_num = int(last_folder.split('_')[-1]) + 1
else:
    df = pd.DataFrame(columns=['folder_name', 'folder_path', 'num_frames'])
    start_num = 0

s = 'http://localhost:4747/video"'
# Initialize camera
cap = cv2.VideoCapture(s)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

if not cap.isOpened():
    print("Error: Could not open camera")
    exit()

try:
    for folder_num in tqdm(range(NUM_FOLDERS), desc='Folders'):
        folder_name = f'gesture_2_{folder_num:03d}'
        folder_path = os.path.join(dataset_dir, folder_name)
        os.makedirs(folder_path, exist_ok=True)
        
        print(f"\nCapturing frames for folder {folder_num + 1}/30")
        print("Press SPACE to start capturing...")
        
        while True:
            ret, frame = cap.read()
            cv2.imshow('Preview', frame)
            if cv2.waitKey(1) & 0xFF == ord(' '):
                break
        
        # Capture frames
        for frame_num in tqdm(range(NUM_FRAMES), desc='Frames'):
            ret, frame = cap.read()
            if ret:
                frame_path = os.path.join(folder_path, f'frame_{frame_num:03d}.jpg')
                cv2.imwrite(frame_path, frame)
                cv2.imshow('Capturing', frame)
                cv2.waitKey(100)
        
        # Add folder metadata to DataFrame
        new_row = {
            'folder_name': folder_name,
            'folder_path': folder_path,
            'num_frames': NUM_FRAMES
        }
        df.loc[len(df)] = new_row
        
        # Save CSV after each folder
        df.to_csv(csv_file, index=False)
        
        print(f"Completed folder {folder_num + 1}. Press SPACE to continue...")
        while cv2.waitKey(1) & 0xFF != ord(' '):
            pass

finally:
    cap.release()
    cv2.destroyAllWindows()
    print(f"\nDataset creation complete. Metadata saved to {csv_file}")

**Augment the Train Data**

In [ ]:
import cv2
import numpy as np
import os
from tqdm import tqdm
import pandas as pd

def rotate_image(image, angle):
    height, width = image.shape[:2]
    center = (width/2, height/2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, rotation_matrix, (width, height))

def add_gaussian_noise(image, mean=0, sigma=25):
    noise = np.random.normal(mean, sigma, image.shape).astype(np.uint8)
    noisy_image = cv2.add(image, noise)
    return noisy_image

def scale_image(image, scale_factor):
    width = int(image.shape[1] * scale_factor)
    height = int(image.shape[0] * scale_factor)
    return cv2.resize(image, (width, height))

def adjust_brightness(image, factor):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv[:,:,2] = np.clip(hsv[:,:,2] * factor, 0, 255)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

# Load existing CSV
csv_file = 'dataset_metadata.csv'
df = pd.read_csv(csv_file)

# Augmentation parameters
rotations = [15, -15, 30, -30]
scales = [0.8, 1.2]
noise_levels = [10, 25]
brightness_factors = [0.7, 1.3]

# Create directories for augmented data
aug_types = ['rotation', 'scale', 'noise', 'brightness', 'flip']
dataset_dir = 'gesture_dataset'

for idx, row in tqdm(df.iterrows(), desc='Processing folders'):
    folder_path = row['folder_path']
    folder_name = row['folder_name']
    
    # Process each augmentation type
    for aug_type in aug_types:
        if aug_type == 'rotation':
            for angle in rotations:
                aug_folder = f"{folder_name}_rot{angle}"
                aug_path = os.path.join(dataset_dir, aug_folder)
                os.makedirs(aug_path, exist_ok=True)
                
                for frame in range(row['num_frames']):
                    img_path = os.path.join(folder_path, f'frame_{frame:03d}.jpg')
                    img = cv2.imread(img_path)
                    if img is not None:
                        rotated = rotate_image(img, angle)
                        cv2.imwrite(os.path.join(aug_path, f'frame_{frame:03d}.jpg'), rotated)
                
                # Add to DataFrame
                new_row = {
                    'folder_name': aug_folder,
                    'folder_path': aug_path,
                    'num_frames': row['num_frames']
                }
                df = df.append(new_row, ignore_index=True)
        
        elif aug_type == 'scale':
            for scale in scales:
                aug_folder = f"{folder_name}_scale{scale}"
                aug_path = os.path.join(dataset_dir, aug_folder)
                os.makedirs(aug_path, exist_ok=True)
                
                for frame in range(row['num_frames']):
                    img_path = os.path.join(folder_path, f'frame_{frame:03d}.jpg')
                    img = cv2.imread(img_path)
                    if img is not None:
                        scaled = scale_image(img, scale)
                        cv2.imwrite(os.path.join(aug_path, f'frame_{frame:03d}.jpg'), scaled)
                
                df = df.append({
                    'folder_name': aug_folder,
                    'folder_path': aug_path,
                    'num_frames': row['num_frames']
                }, ignore_index=True)
        
        elif aug_type == 'noise':
            for sigma in noise_levels:
                aug_folder = f"{folder_name}_noise{sigma}"
                aug_path = os.path.join(dataset_dir, aug_folder)
                os.makedirs(aug_path, exist_ok=True)
                
                for frame in range(row['num_frames']):
                    img_path = os.path.join(folder_path, f'frame_{frame:03d}.jpg')
                    img = cv2.imread(img_path)
                    if img is not None:
                        noisy = add_gaussian_noise(img, sigma=sigma)
                        cv2.imwrite(os.path.join(aug_path, f'frame_{frame:03d}.jpg'), noisy)
                
                df = df.append({
                    'folder_name': aug_folder,
                    'folder_path': aug_path,
                    'num_frames': row['num_frames']
                }, ignore_index=True)
        
        elif aug_type == 'brightness':
            for factor in brightness_factors:
                aug_folder = f"{folder_name}_bright{factor}"
                aug_path = os.path.join(dataset_dir, aug_folder)
                os.makedirs(aug_path, exist_ok=True)
                
                for frame in range(row['num_frames']):
                    img_path = os.path.join(folder_path, f'frame_{frame:03d}.jpg')
                    img = cv2.imread(img_path)
                    if img is not None:
                        bright = adjust_brightness(img, factor)
                        cv2.imwrite(os.path.join(aug_path, f'frame_{frame:03d}.jpg'), bright)
                
                df = df.append({
                    'folder_name': aug_folder,
                    'folder_path': aug_path,
                    'num_frames': row['num_frames']
                }, ignore_index=True)
        
        elif aug_type == 'flip':
            aug_folder = f"{folder_name}_flip"
            aug_path = os.path.join(dataset_dir, aug_folder)
            os.makedirs(aug_path, exist_ok=True)
            
            for frame in range(row['num_frames']):
                img_path = os.path.join(folder_path, f'frame_{frame:03d}.jpg')
                img = cv2.imread(img_path)
                if img is not None:
                    flipped = cv2.flip(img, 1)  # Horizontal flip
                    cv2.imwrite(os.path.join(aug_path, f'frame_{frame:03d}.jpg'), flipped)
            
            df = df.append({
                'folder_name': aug_folder,
                'folder_path': aug_path,
                'num_frames': row['num_frames']
            }, ignore_index=True)

# Save updated CSV
df.to_csv(csv_file, index=False)
print("Augmentation complete!")

In [5]:
import cv2
import os
from pathlib import Path

def verify_and_fix_paths(base_dir, csv_file):
    # Convert to absolute path
    abs_base_dir = Path(base_dir).resolve()
    print(f"Checking paths in: {abs_base_dir}")
    
    # Load CSV
    df = pd.read_csv(csv_file)
    
    # Verify each folder exists
    for idx, row in df.iterrows():
        video_id = row['video_id']
        folder_path = abs_base_dir / str(video_id)
        
        if not folder_path.exists():
            print(f"Missing folder: {folder_path}")
            continue
            
        # Verify frames exist
        for frame in range(row['frames']):
            frame_path = folder_path / f'frame_{frame:03d}.jpg'
            if not frame_path.exists():
                print(f"Missing frame: {frame_path}")
                
    return abs_base_dir

# Usage
dataset_dir = './Gesture Dataset'
csv_file = 'dataset_metadata.csv'

# Verify paths before processing
abs_dataset_dir = verify_and_fix_paths(dataset_dir, csv_file)

# Update image reading code with absolute paths
for frame in range(frames):
    img_path = os.path.join(abs_dataset_dir, str(video_id), f'frame_{frame:03d}.jpg')
    img = cv2.imread(str(img_path))  # Convert Path to string
    if img is None:
        print(f"Failed to read: {img_path}")
        continue
    # Process image...

Checking paths in: /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset
Missing folder: /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_000
Missing folder: /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_001
Missing folder: /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_002
Missing folder: /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_003
Missing folder: /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_004
Missing folder: /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_005
Missing folder: /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_006
Missing folder: /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_007
Missing folder: /mnt/Main Drive/Codes

[ WARN:0@650.769] global loadsave.cpp:241 findDecoder imread_('/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_2_029/frame_000.jpg'): can't open/read file: check file path/integrity
[ WARN:0@650.769] global loadsave.cpp:241 findDecoder imread_('/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_2_029/frame_001.jpg'): can't open/read file: check file path/integrity
[ WARN:0@650.769] global loadsave.cpp:241 findDecoder imread_('/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_2_029/frame_002.jpg'): can't open/read file: check file path/integrity
[ WARN:0@650.769] global loadsave.cpp:241 findDecoder imread_('/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/Gesture Dataset/gesture_2_029/frame_003.jpg'): can't open/read file: check file path/integrity
[ WARN:0@650.770] global loadsave.cpp:241 findDecoder imread_('/mnt/Main Drive/Codes/Deep Learning/Gesture_contr

In [ ]:
import cv2
import numpy as np
import os
from tqdm import tqdm
import pandas as pd

def load_dataset_info(csv_path):
    return pd.read_csv(csv_path)

def save_augmented_info(df, video_id, label, label_id, aug_type, aug_param, frames, folder_path):
    return {
        'video_id': f"{video_id}_{aug_type}_{aug_param}",
        'label': label,
        'label_id': label_id,
        'frames': frames,
        'folder_path': folder_path
    }

# Load existing CSV
csv_file = 'dataset_metadata.csv'
df = load_dataset_info(csv_file)
new_records = []

# Augmentation parameters
rotations = [15, -15, 30, -30]
scales = [0.8, 1.2]
noise_levels = [10, 25]
brightness_factors = [0.7, 1.3]

dataset_dir = './Gesture Dataset'

# Process each video in dataset
for idx, row in tqdm(df.iterrows(), desc='Processing videos'):
    video_id = row['video_id']
    label = row['label']
    label_id = row['label_id']
    frames = row['frames']
    
    base_folder = os.path.join(dataset_dir, str(video_id))
    
    # Apply each augmentation
    for angle in rotations:
        aug_folder = f"{video_id}_rot{angle}"
        aug_path = os.path.join(dataset_dir, aug_folder)
        os.makedirs(aug_path, exist_ok=True)
        
        for frame in range(frames):
            img_path = os.path.join(base_folder, f'frame_{frame:03d}.jpg')
            img = cv2.imread(img_path)
            if img is not None:
                rotated = rotate_image(img, angle)
                cv2.imwrite(os.path.join(aug_path, f'frame_{frame:03d}.jpg'), rotated)
        
        new_records.append(save_augmented_info(
            df, video_id, label, label_id, 'rot', angle, frames, aug_path
        ))

    # Similar blocks for other augmentations...
    # Scale
    for scale in scales:
        aug_folder = f"{video_id}_scale{scale}"
        aug_path = os.path.join(dataset_dir, aug_folder)
        os.makedirs(aug_path, exist_ok=True)
        
        for frame in range(frames):
            img_path = os.path.join(base_folder, f'frame_{frame:03d}.jpg')
            img = cv2.imread(img_path)
            if img is not None:
                scaled = scale_image(img, scale)
                cv2.imwrite(os.path.join(aug_path, f'frame_{frame:03d}.jpg'), scaled)
        
        new_records.append(save_augmented_info(
            df, video_id, label, label_id, 'scale', scale, frames, aug_path
        ))

    # Noise
    for sigma in noise_levels:
        aug_folder = f"{video_id}_noise{sigma}"
        aug_path = os.path.join(dataset_dir, aug_folder)
        os.makedirs(aug_path, exist_ok=True)
        
        for frame in range(frames):
            img_path = os.path.join(base_folder, f'frame_{frame:03d}.jpg')
            img = cv2.imread(img_path)
            if img is not None:
                noisy = add_gaussian_noise(img, sigma=sigma)
                cv2.imwrite(os.path.join(aug_path, f'frame_{frame:03d}.jpg'), noisy)
        
        new_records.append(save_augmented_info(
            df, video_id, label, label_id, 'noise', sigma, frames, aug_path
        ))

# Create new DataFrame with augmented data
augmented_df = pd.DataFrame(new_records)
final_df = pd.concat([df, augmented_df], ignore_index=True)

# Save updated CSV
final_df.to_csv('dataset_augmented.csv', index=False)
print("Augmentation complete!")

Processing videos: 0it [00:00, ?it/s][ WARN:0@213.449] global loadsave.cpp:241 findDecoder imread_('./Gesture Dataset/gesture_000/frame_000.jpg'): can't open/read file: check file path/integrity
[ WARN:0@213.449] global loadsave.cpp:241 findDecoder imread_('./Gesture Dataset/gesture_000/frame_001.jpg'): can't open/read file: check file path/integrity
[ WARN:0@213.450] global loadsave.cpp:241 findDecoder imread_('./Gesture Dataset/gesture_000/frame_002.jpg'): can't open/read file: check file path/integrity
[ WARN:0@213.450] global loadsave.cpp:241 findDecoder imread_('./Gesture Dataset/gesture_000/frame_003.jpg'): can't open/read file: check file path/integrity
[ WARN:0@213.450] global loadsave.cpp:241 findDecoder imread_('./Gesture Dataset/gesture_000/frame_004.jpg'): can't open/read file: check file path/integrity
[ WARN:0@213.450] global loadsave.cpp:241 findDecoder imread_('./Gesture Dataset/gesture_000/frame_005.jpg'): can't open/read file: check file path/integrity
[ WARN:0@213.45

Augmentation complete!


In [1]:
import os
import cv2
import pandas as pd
import albumentations as A
from tqdm import tqdm

# Load the original train_data DataFrame
train_data = pd.read_csv('archive/Train copy.csv')

# Directory containing the original frames
main_dir = '/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive/Train'
augmented_main_dir = '/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/augmented/Train'

# Create the augmented main directory if it doesn't exist
if not os.path.exists(augmented_main_dir):
    os.makedirs(augmented_main_dir)

# Define individual transformations with specific names
transformations = [
    (A.ShiftScaleRotate(shift_limit=0, scale_limit=0.2, rotate_limit=0, p=1.0), 'Scaled'),  # Only scaling
    (A.ShiftScaleRotate(shift_limit=0, scale_limit=0, rotate_limit=15, p=1.0), 'Rotated'),  # Only rotation
    (A.Affine(shear=15, p=0.5), 'Sheared'),  # Shearing
    (A.RandomBrightnessContrast(p=0.5), 'BrightnessContrast'),  # Brightness and contrast
    (A.GaussNoise(var_limit=(10.0, 50.0), p=0.5), 'GaussNoise'),  # Gaussian noise
]

def augment_and_save_frames(input_dir, output_dir, transform):
    # List all frame files and sort them
    frames = sorted([f for f in os.listdir(input_dir) if f.endswith('.jpg')])
    for frame_file in frames:
        frame_path = os.path.join(input_dir, frame_file)
        # Read the image
        image = cv2.imread(frame_path)
        if image is None:
            print(f"Warning: Frame {frame_path} could not be read and will be skipped.")
            continue
        # Apply the augmentation
        augmented = transform(image=image)
        augmented_frame = augmented['image']
        # Define the output path for the augmented frame
        output_frame_path = os.path.join(output_dir, frame_file)
        # Save the augmented frame
        cv2.imwrite(output_frame_path, augmented_frame)

# Initialize a list to store the augmented data information
augmented_data = []

# Iterate through each folder in the main directory
for folder_name in os.listdir(main_dir):
    folder_path = os.path.join(main_dir, folder_name)
    if not os.path.isdir(folder_path):
        continue

    # Apply each transformation and save the augmented frames in separate folders
    for transform, transform_name in transformations:
        # Create a subdirectory for the current transformation
        augmented_folder_path = os.path.join(augmented_main_dir, f"{folder_name}_{transform_name}")
        if not os.path.exists(augmented_folder_path):
            os.makedirs(augmented_folder_path)
        
        # Apply the transformation and save the augmented frames
        augment_and_save_frames(folder_path, augmented_folder_path, transform)
        
        # Get the label and other details from the original train_data
        video_id = int(folder_name)
        if video_id not in train_data['video_id'].values:
            print(f"Warning: video_id {video_id} not found in train_data. Skipping folder {folder_name}.")
            continue
        
        label = train_data.loc[train_data['video_id'] == video_id, 'label'].values[0]
        label_id = train_data.loc[train_data['video_id'] == video_id, 'label_id'].values[0]
        shape = train_data.loc[train_data['video_id'] == video_id, 'shape'].values[0]
        format = 'JPEG'  # Assuming the format is constant
        
        # Append the augmented data to the list
        augmented_data.append({
            'video_id': f"{folder_name}_{transform_name}",
            'label': label,
            'frames': len(os.listdir(augmented_folder_path)),
            'label_id': label_id,
            'shape': shape,
            'format': format
        })

# Create a DataFrame for the augmented data
augmented_df = pd.DataFrame(augmented_data)

# Save the augmented DataFrame to a CSV file
augmented_df.to_csv('augmented_train_data.csv', index=False)

print("Transformations applied and saved to", augmented_main_dir)
print("Augmented data saved to augmented_train_data.csv")

INFO:albumentations.check_version:A new version of Albumentations is available: 1.4.14 (you have 1.4.7). Upgrade using: pip install --upgrade albumentations


Transformations applied and saved to /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/augmented/Train
Augmented data saved to augmented_train_data.csv


In [17]:
import os
import pandas as pd

# Load the original train_data DataFrame
original_csv_path = '/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive/Validation.csv'
train_data = pd.read_csv(original_csv_path)

# Directory containing the original frames
val_dir = '/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive/Val'

# List all folders in the val directory
val_folders = [folder for folder in os.listdir(val_dir) if os.path.isdir(os.path.join(val_dir, folder))]

# Convert folder names to integers (assuming folder names are integers)
val_folder_ids = [int(folder) for folder in val_folders]

# Filter the original DataFrame to include only the rows corresponding to the folders in the val directory
filtered_data = train_data[train_data['video_id'].isin(val_folder_ids)]

# Save the filtered DataFrame to a new CSV file
filtered_csv_path = '/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/filtered_val_data.csv'
filtered_data.to_csv(filtered_csv_path, index=False)

print(f"Filtered data saved to {filtered_csv_path}")

Filtered data saved to /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/filtered_val_data.csv
